   ... custom `sim_forward` function is essentially just a safety wrapper...
   It uses `assert` to guarantee your pointers aren't completely broken before
   handing the ... MuJoCo's built-in `mj_forward` function.
   
WHAT `mj_forward` ACTUALLY DOES:
   Think of it as the "RECALCULATE" or "REFRESH" BUTTON for your physics engine.
   When you manually change a state in your C code--like restting the robot's
   joint angles for a new trial, or teleporting the plug to a new starting 
   position--MuJoCo does not instanly update the 3D world coordinates of all the
   connected parts. TO save processing power, it waits until you explicitly
   tell it do the math.

   Calling `mj_forward` forces MuJoCo to run its forward kinematics pipeline.
   It takes the joint angles of positions you just set, and crunches the math to
   figure out exactly where your `plug_tip` and `socket_mouth` are currently
   located in 3D space, WITHOUT ADVANCING THE SIMULATION CLOCK forward.

   THE GOLDEN RULE: 
   - Use `mj_step` when you to move forward both time and physics to happen.
   - Use `mj_forward` when you manually moved something in the code and need
     the engine to update its geometry arrays (like `data->site_xpos`) so your
     evaluator doesn't accidentallly grade the trial using stale data
     from the previous frame.


---

   The bug was not your actuator test logic in the end. The imported full AIC
   MuJoCO scene is dynamically unstable when advanced with `mj_step`, even
   before meaningful control. THe giveaway was passive stepping:

   WARNING: Nan, Inf or huge value in QACC (Joint Acceleration) at DOF 62
   ...

   This means the cable/free-joint/plugin branch is exploding under standalone
   MuJoCo dynamics. The actuators were actually wired: all 7 can move joints, 
   ... but in the original scene every actuator probe was jmarked UNSTABLE.

---
Methodology used:
   - Reproduced the issue with a passive `mj_step` test, separating "scene
     dynamics unstable" from ...
   - .... COmpared original copied scene vs. derived rollout scene under the
     same probe.
   - Replaced actuators with explicit `<position>` actuators and added `plug_tip`
     as a robot tool site so evaluator plumbing still works.

   ...
   
   Practical takeaway: use the original full scene for static evaluator/site
   geometry, but use the derived rollout scene for controller/RL work. It is 
   less physically rich becuase the cable is removed, but it 


---

   Passive stepping is what happens when you let MuJoCo simulation run while
   sending absolutely zero control signals to your robot's motors. Think of
   ... In your C code, this means you are calling `mj_step(model, data)` in a 
   loop, but you haven't written any values into the `data->ctrl` array. You are
   acting purely as an observer, watching the physical world take its natural
   course.

   Even without active motor forces, a lot of intense math is sitll happening 
   under the hood during a passive step. MuJoCo calculates the effects of gravity
   pulling the robot down, the built-in friction (damping) inside the joints 
   resisting movement, the springiness (stifness) of any limits, and the complex
   collision forces when the plug hits the rack or the floor. 
   
   Running a passive step test is usually the first thing roboticists do when a 
   newly imported XML model just to make sure the robot falls naturally and 
   doesn't instantly explode into the sky due to bad collision geometry.

   When you want to stop being passive and start driving the robot, you use 
   actuators, which brings us to the `<general>` tag in your XML. The 
   `<general>` tag is MuJoCo's raw fundamental muscle. When you send a control
   signal (like a `1.0`) to a `<general>` actuator from your code, you are
   telling MuJoCo to apply exactly that amount of raw force or torque directly
   to the joint. It is "dumb" in the best way possible: it doesn't know where
   the joint currently is or where you want it to go; it just pushes with the
   exact electrical effort you commanded, leaving the complex math or 
   target-tracking entirely up to your custom staged controller.

   The `<position>` tag, on the other hand, is a "smart" actuator--essentially a
   built-in servo motor. Instead of accepting raw force commands, it accepts a
   target destination. If you send a `1.0` ot a `<position>` actuator on a 
   rotational joint, you are telling it, "Move to 1.0 radians and hold it there,"
   Under the hood, MuJoCo automatically runs a Proportional-Derivatice control
   algorithm, constantly calculating and applying the exact right amount of raw
   force needed to reach that angle and stop. It is a highly convenient shortcut
   if you want the robot to hold a specific pose without having to hand-code the
   torque calculations yourself!

---

- Keeps the cable branch.
- Keeps the cable elasticity plugin, but reduces stiffness heavily.
- Increases cable inertias/mass
- Adds damping/armature to cable joints.
- Disables flexible cable-chain collision geoms, but keeps the plug body.
- Use position actuators...




### MuJoCo QACC Error (Joint Acceleration Instability)
   ...
   - WHAT IT MEANS: `QACC` stands for JOINT ACCELERATION. This warning appears
     when a physics calculation calculates an impossible/infinite acceleration
     in a specific degree of freedom (DOF), leading to the robot clipping, 
     cartwheeling, or breaking the physics grid.
   - HOW TO FIX IT:
      - INCREASE DAMPING/ARMATURE: Add `damping` or `armature` parameters to 
        your joints in your XML/MJCF file to abosrb abrupt force or torque
        applications.
      - REDUCE TIME STEP: Try lowering simulation timestep (... `<option timestep="0.002"/>`)
        to prevent overshooting during mmathematical integration.
      - CHECK ACTUATORS: Verify that your neural network policy or PID 
        controller is not applying excessively large, erratic control actions
        (like `-ctrl` signals) that send the joint spinning beyond its physical
        limits.  

   ... DAMPING and ARMATURE ... completely different parameters in a physics or
   robotics simulation. While both affect how a joint moves, they represent 
   different physical properties and serve different functions.

   ...
   - DAMPING: This is a force that RESISTS MOTION OR ABSORBS ENERGY. It acts like
     a shock absorber or swimming through water. High damping slows down the 
     joint and brings the system to a resting state quickly, preventing it from
     oscillating endlessly.
   - ARMATURE: This is the INERTIAL MASS OF THE MOTOR'S MOVING PARTS (like the 
     rotor). It represents the motor's natural resistance to chaning its speed
     (acceleration or deceleration). A high armature makes the joint feel "heavy"  
     , making it take longer to start moving or stop.

---

   This is the mathematical hearteat of your robot's movement! While your
   `evaluator.c` is the judge and `sim.c` is the physics world, this 
   `trajectory.c` ile is the MOTION DIRECTOR. Its entire purpose is to prevent
   your robot from ripping itself apart when moving from Point A to Point B.

   ... how these pieces fit together to create robotic trajectories:

---
1. THE SAFETY NET: `traj_clamp`
   In robotics, math can sometimes spit out numbers like `1.000001` due to 
   floating-point drift. If you feed that into an algorithm expecting a strict
   percentage between `0.0` and `1.0`, the system can crash or glitch. 
   `traj_clamp` simply takes any number and strictly confines it within a floor
   (`lo`) and a ceiling (`hi`). It is pure defensive programming.

2. THE STRAIGHT LINE: `traj_lerp`
   "Lerp" stands for LINEAR INTERPOLATION. Imagine your robot's elbow is 
   currently at 10-deg (a), and you want it to move to 90-deg (b). You define a
   time percentage $t$ that goes from 0.0 to 1.0.

   - When $t = 0.0$, the math turns 10-deg
   ... t = 0.5 ... 50-deg
   ... t = 1.0 ... 90-deg

3. THE MAGIC OF ROBOTICS: `traj_smoothstep`
   If you only use a straight line (`lerp`) to move a robot arm, the robot will
   instantly snap into maximum speed at $t = 0$, and violently crash to a halt 
   at $t = 1$. This infinite spike in acceleration (called "jerk") damages real
   gearboxes and makes simulated physics explode.

   `traj_smoothstep` fixes this by taking your straight time $t$ and warping it
   using a cubic polynomial... This creates an "S-curve." When time starts, the
   curbe ramps up slowly (EASE-IN), reaches top speed in the middle, and then 
   gently decelerates to a stoop at the end (EASE-OUT).

4. THE PUPPETEER: `traj_lerp_array`
   A robot arm doesn't just have one motor; it has 6 or 7. If you move them all
   at different times, the arm flails unpredictably. `traj_lerp_array` takes an 
   array of all your goal joint angles, and your current time $t$. First, it
   passes $t$ through the `traj_smoothstep` S-curve. Then, it applies that
   smoothed percentage to every single motor at once. This guarantees that all 
   motors start moving simultaneouly, accelerate smoothly, and arrive at their
   exact goal angles at the exact same millisecond, producing a beautiful, 
   coordinated robotic motion.

   ... 


